# Meta-encoder + EXACT (hardcoded) decoder -- set-prediction training

Replaces the learned autoregressive bit-decoder with a parameter-free **exact decoder**:
the encoder emits NUM_SLOTS=64 soft strings (16 targets x 4), and the loss is the
exact log-likelihood of the true memorized strings under the product-Bernoulli
**mixture** of those slots. Training forces the slots to *become* the strings, in
one parallel forward pass (no per-bit recurrence).

- Parallel depth (the "number of iterations") = ENCODER_LAYERS + SLOT_LAYERS.
- Now: ENCODER_LAYERS=1, SLOT_LAYERS=1. Set ENCODER_LAYERS=2 to test more depth.
- 48 epochs, Kaggle GPU (DataParallel + AMP + torch.compile). Runs main() on import.
- Validated on CPU: backprop through the exact decoder works; overfitting 8 MLPs
  reaches 16/16 exact recovery. This is the generalization run.
- Metrics per epoch: recover(exact) = true strings appearing as a binarized slot;
  recover(MLP-verified) = those also accepted by the MLP.

In [1]:
# =========================================================
# META-ENCODER + EXACT (HARDCODED) DECODER  -- set prediction
# =========================================================
# Idea: replace the learned autoregressive bit-decoder with an EXACT, parameter
# -free decoder. The encoder emits NUM_SLOTS soft strings (= N_TARGETS x SLOT_REP
# slots). The "decoder" is the closed-form log-likelihood of the true memorized
# strings under the product-Bernoulli mixture defined by those slots -- the exact
# logprob the autoregressive decoder used to approximate. Training pushes the
# slots to BECOME the strings, in ONE parallel forward pass (no per-bit recurrence).
#
# Parallel depth (the "number of iterations") = ENCODER_LAYERS + SLOT_LAYERS.
# Start at 1 encoder layer; SLOT_LAYERS=1. Both are configurable below.
# =========================================================
import os, glob, time, math, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | GPU Count: {torch.cuda.device_count()}")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DATA_DIR = "/kaggle/input/notebooks/emanuelruzak/lpe9datagen48bit-16strings-ipynb/dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/working/dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "dataset"                       # local fallback

CKPT_PATH = "/kaggle/working/meta_exact_decoder_48bit.pt"
if not os.path.isdir(os.path.dirname(CKPT_PATH)):
    CKPT_PATH = "meta_exact_decoder_48bit.pt"

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
SMOKE = bool(int(os.environ.get("SMOKE", "0")))   # local quick test

D, H, N_TARGETS = 48, 128, 16
SLOT_REPEAT = 4
NUM_SLOTS   = N_TARGETS * SLOT_REPEAT             # 64
D_MODEL, NHEAD = 256, 8
ENCODER_LAYERS = 8                                # <- 1 now; set 2 to test depth
SLOT_LAYERS    = 1                                # parallel "read the list" depth

BATCH_SIZE = 32                                   # MLPs per step (no per-target expand now)
EPOCHS     = 48
LR, ETA_MIN, GRAD_CLIP = 3e-4, 1e-5, 1.0
USE_COMPILE = True

KS = [16, 32, 64, 128, 256, 512, 1024]            # secrets-in-top-k recovery budgets (fair vs GCG)

if SMOKE:
    BATCH_SIZE, EPOCHS, USE_COMPILE = 8, 2, False

# ---------------------------------------------------------
# MODULES (RMSNorm, ReLU^2, no dropout) -- same primitives as the original
# ---------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class CustomEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=1024):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=0.0)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
    def forward(self, src):
        n = self.norm1(src)
        a, _ = self.self_attn(n, n, n)
        src = src + a
        n = self.norm2(src)
        src = src + self.linear2(torch.square(F.relu(self.linear1(n))))
        return src

class SlotLayer(nn.Module):
    """Slot queries: self-attention among slots (diversity) + cross-attention to
    the neuron memory + ReLU^2 FFN. No causal mask -- slots are an unordered set."""
    def __init__(self, d_model, nhead, dim_feedforward=1024):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=0.0)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=0.0)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.norm3 = RMSNorm(d_model)
    def forward(self, slots, memory):
        n = self.norm1(slots)
        a, _ = self.self_attn(n, n, n)
        slots = slots + a
        n = self.norm2(slots)
        a, _ = self.cross_attn(n, memory, memory)
        slots = slots + a
        n = self.norm3(slots)
        slots = slots + self.linear2(torch.square(F.relu(self.linear1(n))))
        return slots

# ---------------------------------------------------------
# MODEL: weights -> 64 soft strings (slots)
# ---------------------------------------------------------
class MetaSetEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL, nhead=NHEAD,
                 enc_layers=ENCODER_LAYERS, slot_layers=SLOT_LAYERS, num_slots=NUM_SLOTS):
        super().__init__()
        self.weight_proj = nn.Sequential(nn.Linear(51, d_model), RMSNorm(d_model))
        self.encoder_layers = nn.ModuleList(
            [CustomEncoderLayer(d_model, nhead) for _ in range(enc_layers)])
        self.encoder_norm = RMSNorm(d_model)

        self.num_slots = num_slots
        self.slot_queries = nn.Parameter(torch.randn(num_slots, d_model) * 0.02)
        self.slot_pos = nn.Parameter(torch.randn(num_slots, d_model) * 0.02)  # break symmetry
        self.slot_layers = nn.ModuleList(
            [SlotLayer(d_model, nhead) for _ in range(slot_layers)])
        self.slot_norm = RMSNorm(d_model)
        self.slot_head = nn.Linear(d_model, D)        # 48-bit logits per slot

    def encode(self, w1, b1, w2, b2):
        b1e = b1.unsqueeze(-1)
        w2e = w2.transpose(1, 2)
        b2e = b2.unsqueeze(-1).expand(-1, H, 1)
        tokens = torch.cat([w1, b1e, w2e, b2e], dim=-1)   # (B,128,51)
        m = self.weight_proj(tokens)
        for layer in self.encoder_layers:
            m = layer(m)
        return self.encoder_norm(m)                        # (B,128,256)

    def forward(self, w1, b1, w2, b2):
        memory = self.encode(w1, b1, w2, b2)
        B = memory.size(0)
        slots = (self.slot_queries + self.slot_pos).unsqueeze(0).expand(B, -1, -1)
        for layer in self.slot_layers:
            slots = layer(slots, memory)
        slots = self.slot_norm(slots)
        return self.slot_head(slots)                       # (B,NUM_SLOTS,48) logits

# ---------------------------------------------------------
# EXACT DECODER LOSS: NLL of the true strings under the slot mixture
#   p(s) = (1/S) sum_l prod_i Bernoulli(s_i ; sigmoid(z_{l,i}))
#   log p(s) = logsumexp_l [ -log S + sum_i logsigmoid((2 s_i - 1) z_{l,i}) ]
# Exact and fully differentiable; permutation-invariant over slots (no matching).
# ---------------------------------------------------------
def exact_decoder_nll(slot_logits, targets):
    # slot_logits (B,S,48) ; targets (B,T,48) in {0,1}
    S = slot_logits.size(1)
    sign = (2.0 * targets - 1.0).unsqueeze(2)            # (B,T,1,48)
    z = slot_logits.unsqueeze(1)                         # (B,1,S,48)
    logp = F.logsigmoid(sign * z).sum(-1)               # (B,T,S) per-slot logprob
    log_mix = torch.logsumexp(logp - math.log(S), dim=2)  # (B,T)
    return -log_mix.mean()

# ---------------------------------------------------------
# DATASET: one item = one MLP + its 16 target strings
# ---------------------------------------------------------
class MLPSetDataset(Dataset):
    def __init__(self, files):
        super().__init__()
        self.cache = []
        for f in files:
            try:
                self.cache.append(torch.load(f, map_location="cpu", weights_only=False))
            except Exception as e:
                print(f"skip {f}: {e}")
        self.mpc = self.cache[0]["W1"].shape[0] if self.cache else 0
        self.total = len(self.cache) * self.mpc
    def __len__(self):
        return self.total
    def _raw(self, idx):
        ch = self.cache[idx // self.mpc]; mi = idx % self.mpc
        w1 = ch["W1"][mi].float(); b1 = ch["b1"][mi].float()
        w2 = ch["W2"][mi].float(); b2 = ch["b2"][mi].float()
        targets = ch["templates"][mi][:, 1:].float()      # (16,48) drop BOS col
        return w1, b1, w2, b2, targets
    def __getitem__(self, idx):
        return self._raw(idx)
    def get_batch1(self, idx, dev):
        w1, b1, w2, b2, t = self._raw(idx)
        return (w1[None].to(dev), b1[None].to(dev), w2[None].to(dev),
                b2[None].to(dev), t.to(dev))

def collate(batch):
    w1 = torch.stack([b[0] for b in batch])
    b1 = torch.stack([b[1] for b in batch])
    w2 = torch.stack([b[2] for b in batch])
    b2 = torch.stack([b[3] for b in batch])
    t  = torch.stack([b[4] for b in batch])
    return w1, b1, w2, b2, t

# ---------------------------------------------------------
# RECOVERY EVAL: rank candidate strings by EXACT mixture probability ->
# secrets-in-top-k (a fair, false-positive-aware comparison with GCG, which is
# scored at a fixed candidate budget). Also report the sorted per-secret NLL.
# ---------------------------------------------------------
def _unwrap(m):
    while isinstance(m, nn.DataParallel) or hasattr(m, "_orig_mod"):
        m = m.module if isinstance(m, nn.DataParallel) else m._orig_mod
    return m

def _bits_to_int(bits):
    shifts = torch.arange(bits.shape[-1], dtype=torch.int64, device=bits.device)
    return (bits.to(torch.int64) << shifts).sum(-1)

@torch.no_grad()
def topk_recovery(z, secrets, ks=KS, r=8, cap=4096, chunk=512):
    # z: (S,D) slot logits for ONE organism; secrets: (T,D).
    # Build a candidate pool (each slot's argmax + all flips of its r least-confident bits),
    # rank by EXACT mixture log-prob, and count how many secrets fall in the top-k.
    z = z.float(); S, Dd = z.shape; r = min(r, Dd)
    am  = (z > 0).float()                                          # per-slot argmax string
    low = z.abs().topk(r, dim=1, largest=False).indices           # r least-confident bits per slot
    P = 1 << r
    pats = ((torch.arange(P, device=z.device).unsqueeze(1) >> torch.arange(r, device=z.device)) & 1).float()
    cand = am.unsqueeze(1).expand(S, P, Dd).clone()
    idx  = low.unsqueeze(1).expand(S, P, r)
    fl   = pats.unsqueeze(0).expand(S, P, r)
    cand.scatter_(2, idx, (cand.gather(2, idx) - fl).abs())       # XOR each pattern into the low bits
    base = F.logsigmoid(z.abs()).sum(1, keepdim=True)             # per-slot argmax log-prob
    cost = (z.abs().gather(1, low).unsqueeze(1) * fl).sum(2)      # log-prob lost per flip
    prop = (base - cost).reshape(-1)                             # per-slot log-prob of each candidate
    cand = cand.reshape(S * P, Dd)
    if cand.shape[0] > cap:                                       # keep the strongest `cap` proposals
        cand = cand[prop.topk(cap).indices]
    logS = math.log(S); lps = []                                 # exact mixture log-prob, chunked
    for i in range(0, cand.shape[0], chunk):
        c = cand[i:i+chunk]; sign = 2.0 * c - 1.0
        lp = F.logsigmoid(sign.unsqueeze(1) * z.unsqueeze(0)).sum(-1)       # (m,S)
        lps.append(torch.logsumexp(lp - logS, dim=1))
    logp = torch.cat(lps); ci = _bits_to_int(cand)
    uniq, inv = torch.unique(ci, return_inverse=True)            # dedupe, keep best log-prob per string
    ulp = torch.full((uniq.numel(),), float("-inf"), device=z.device)
    ulp.scatter_reduce_(0, inv, logp, reduce="amax", include_self=True)
    ranked = uniq[ulp.argsort(descending=True)]                 # strings best-first
    ti = _bits_to_int(secrets)
    return {k: int(torch.isin(ti, ranked[:k]).sum()) for k in ks}

@torch.no_grad()
def nll_profile(z, secrets):
    # per-secret NLL under the slot mixture, sorted ascending (best..worst)
    z = z.float(); S = z.size(0)
    sign = (2.0 * secrets - 1.0).unsqueeze(1)                     # (T,1,D)
    logp = F.logsigmoid(sign * z.unsqueeze(0)).sum(-1)           # (T,S)
    nll  = -(torch.logsumexp(logp - math.log(S), dim=1))        # (T,)
    return nll.sort().values

@torch.no_grad()
def eval_metrics(model, dataset, n_mlps, ks=KS):
    net = _unwrap(model); was = net.training; net.eval()
    agg = {k: 0.0 for k in ks}; nlls = []; counted = 0
    for gi in range(min(n_mlps, dataset.total)):
        w1, b1, w2, b2, targets = dataset.get_batch1(gi, device)
        amp = (device.type == "cuda")
        with autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            z = net(w1, b1, w2, b2)[0].float()                   # (S,48) slot logits
        hits = topk_recovery(z, targets, ks)
        for k in ks: agg[k] += hits[k]
        nlls.append(nll_profile(z, targets))
        counted += 1
    if was: net.train()
    n = max(1, counted)
    return {k: agg[k] / n for k in ks}, torch.stack(nlls).mean(0)

# ---------------------------------------------------------
# TRAIN
# ---------------------------------------------------------
def main():
    files = glob.glob(os.path.join(DATA_DIR, "*.pt"))
    print(f"Found {len(files)} chunk files in {DATA_DIR}")
    if not files:
        raise FileNotFoundError(f"No .pt files in {DATA_DIR}")
    random.seed(42); random.shuffle(files)
    if SMOKE:
        files = files[:3]
    split = max(1, int(0.9 * len(files)))
    train_files, test_files = files[:split], files[split:] or files[:1]

    train_ds = MLPSetDataset(train_files)
    test_ds = MLPSetDataset(test_files)
    print(f"Total MLPs: train={len(train_ds)} test={len(test_ds)} | slots={NUM_SLOTS} "
          f"enc_layers={ENCODER_LAYERS} slot_layers={SLOT_LAYERS}")

    nw = 0 if SMOKE else 2
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate, num_workers=nw, drop_last=True,
                              pin_memory=(device.type == "cuda"))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collate, num_workers=nw)

    model = MetaSetEncoder().to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    if USE_COMPILE:
        try:
            model = torch.compile(model); print("torch.compile enabled.")
        except Exception as e:
            print(f"compile off: {e}")

    total_steps = max(1, len(train_loader) * EPOCHS)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=ETA_MIN)
    scaler = GradScaler(device.type, enabled=(device.type == "cuda"))
    print(f"steps/epoch={len(train_loader)} total={total_steps}\nTraining...")

    step = 0
    for epoch in range(EPOCHS):
        model.train(); run = 0.0; t0 = time.time()
        for w1, b1, w2, b2, t in train_loader:
            w1, b1, w2, b2, t = [x.to(device) for x in (w1, b1, w2, b2, t)]
            opt.zero_grad(set_to_none=True)
            amp = (device.type == "cuda")
            with autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model(w1, b1, w2, b2)
                loss = exact_decoder_nll(logits.float(), t)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); step += 1
            if step % 200 == 0:
                print(f"  step {step:06d} | nll {loss.item():.4f} | lr {sched.get_last_lr()[0]:.2e}")
        tr = run / max(1, len(train_loader))
        topk, nllp = eval_metrics(model, test_ds, n_mlps=(5 if SMOKE else 30))
        print(f"=== Epoch {epoch+1}/{EPOCHS} | train NLL {tr:.4f} | secrets in top-k: "
              + " ".join(f"top{k}:{topk[k]:.2f}" for k in KS) + f" | {time.time()-t0:.1f}s ===")
        print("    val per-secret NLL by rank (1=best..16=worst): "
              + " ".join(f"{nllp[i]:.1f}" for i in range(N_TARGETS)))
        raw = _unwrap(model)
        torch.save(raw.state_dict(), CKPT_PATH)
    print("Done. ->", CKPT_PATH)

if __name__ == "__main__":
    main()

Using device: cuda | GPU Count: 2
Found 50 chunk files in /kaggle/input/notebooks/emanuelruzak/lpe9datagen48bit-16strings-ipynb/dataset
Total MLPs: train=46080 test=5120 | slots=64 enc_layers=8 slot_layers=1
torch.compile enabled.
steps/epoch=1440 total=69120
Training...


W0622 01:10:00.709000 22 torch/_logging/_internal.py:1204] [0/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:1946: UserWarning: Dynamo does not know how to trace the builtin `_thread.get_ident.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.
  torch._dynamo.utils.warn_once(explanation + "\n" + "\n".join(hints))
W0622 01:10:04.709000 22 torch/_dynamo/convert_frame.py:1676] [3/8] torch._dyna

  step 000200 | nll 33.3060 | lr 3.00e-04
  step 000400 | nll 32.0818 | lr 3.00e-04
  step 000600 | nll 26.4370 | lr 3.00e-04
  step 000800 | nll 25.6982 | lr 3.00e-04
  step 001000 | nll 24.8466 | lr 3.00e-04
  step 001200 | nll 24.4160 | lr 3.00e-04
  step 001400 | nll 24.4259 | lr 3.00e-04
=== Epoch 1/48 | train NLL 27.6440 | secrets in top-k: top16:0.30 top32:0.40 top64:0.47 top128:0.50 top256:0.60 top512:0.70 top1024:0.83 | 103.8s ===
    val per-secret NLL by rank (1=best..16=worst): 14.1 16.0 17.8 19.1 20.4 21.6 22.5 23.2 23.9 24.4 25.4 26.1 27.1 28.1 29.3 31.2
  step 001600 | nll 22.9293 | lr 3.00e-04
  step 001800 | nll 21.1419 | lr 3.00e-04
  step 002000 | nll 20.2748 | lr 2.99e-04
  step 002200 | nll 19.8147 | lr 2.99e-04
  step 002400 | nll 19.8789 | lr 2.99e-04
  step 002600 | nll 19.6742 | lr 2.99e-04
  step 002800 | nll 19.2725 | lr 2.99e-04
=== Epoch 2/48 | train NLL 20.5313 | secrets in top-k: top16:2.43 top32:2.57 top64:2.83 top128:2.97 top256:3.07 top512:3.40 top1024